In [11]:
import json
import os
import cv2
from copy import deepcopy

def extract_twowheeler_data(original_annotation_file, image_folder, output_folder_helmet, output_folder_license, 
                             twowheeler_category_id=3, helmet_category_id=1, license_plate_category_id=2):
    """
    Extracts individual two-wheelers and retains only helmet and license annotations.
    
    Args:
    - original_annotation_file: Path to the original COCO annotation JSON.
    - image_folder: Path to the folder containing images.
    - output_folder_helmet: Folder to save two-wheeler images with only helmet annotations.
    - output_folder_license: Folder to save two-wheeler images with only license plate annotations.
    - twowheeler_category_id: Category ID for two-wheelers.
    - helmet_category_id: Category ID for helmet.
    - license_plate_category_id: Category ID for license plate.
    """

    # Ensure output directories exist
    os.makedirs(output_folder_helmet, exist_ok=True)
    os.makedirs(output_folder_license, exist_ok=True)

    # Load COCO annotations
    with open(original_annotation_file, "r") as f:
        coco_data = json.load(f)

    # Load images and annotations
    annotations = coco_data["annotations"]
    images = {img["id"]: img for img in coco_data["images"]}

    # New annotation structures for helmet and license datasets
    helmet_annotations = []
    license_annotations = []
    new_images_helmet = []
    new_images_license = []

    # Process each image and extract two-wheeler annotations
    for img_data in coco_data["images"]:
        img_id = img_data["id"]
        img_path = os.path.join(image_folder, img_data["file_name"])

        if not os.path.exists(img_path):
            print(f"Warning: {img_path} not found!")
            continue

        img = cv2.imread(img_path)
        
        # Find all two-wheeler annotations in the image
        twowheelers = [ann for ann in annotations if ann["image_id"] == img_id and ann["category_id"] == twowheeler_category_id]
        helmets = {ann["image_id"]: ann for ann in annotations if ann["category_id"] == helmet_category_id}

        for i, tw_ann in enumerate(twowheelers):
            # Get two-wheeler bounding box
            x, y, w, h = map(int, tw_ann["bbox"])
            cropped_img = img[y:y+h, x:x+w]

            new_img_id = f"{img_id}_{i}"  # Unique ID for cropped two-wheeler
            helmet_in_tw = []
            license_in_tw = []

            # Find helmet and license plate annotations inside this two-wheeler
            for ann in annotations:
                if ann["image_id"] != img_id:
                    continue

                ax, ay, aw, ah = map(int, ann["bbox"])  # Original annotation bbox
                
                # Check if annotation is within the two-wheeler region
                if x <= ax and y <= ay and (ax + aw) <= (x + w) and (ay + ah) <= (y + h):
                    new_bbox = [ax - x, ay - y, aw, ah]  # Adjust bbox relative to cropped image

                    if ann["category_id"] == helmet_category_id:
                        new_ann = deepcopy(ann)
                        new_ann["image_id"] = new_img_id
                        new_ann["bbox"] = new_bbox
                        helmet_in_tw.append(new_ann)

                    elif ann["category_id"] == license_plate_category_id:
                        new_ann = deepcopy(ann)
                        new_ann["image_id"] = new_img_id
                        new_ann["bbox"] = new_bbox
                        license_in_tw.append(new_ann)

            # If no helmet is found, estimate the head region
            if not helmet_in_tw:
                est_hx = x + int(0.3 * w)  # Center horizontally
                est_hy = y  # Top of the two-wheeler
                est_hw = int(0.4 * w)  # Approximate head width
                est_hh = int(0.25 * h)  # Approximate head height

                estimated_head_ann = {
                    "image_id": new_img_id,
                    "category_id": 0,  # Still stored as a helmet category but labeled separately
                    "bbox": [est_hx - x, est_hy - y, est_hw, est_hh],
                    "iscrowd": 0
                }
                helmet_in_tw.append(estimated_head_ann)

            # Save cropped images and annotations
            if helmet_in_tw:
                helmet_path = os.path.join(output_folder_helmet, f"{new_img_id}.jpg")
                cv2.imwrite(helmet_path, cropped_img)

                new_images_helmet.append({"id": new_img_id, "file_name": f"{new_img_id}.jpg", "width": w, "height": h})
                helmet_annotations.extend(helmet_in_tw)

            if license_in_tw:
                license_path = os.path.join(output_folder_license, f"{new_img_id}.jpg")
                cv2.imwrite(license_path, cropped_img)

                new_images_license.append({"id": new_img_id, "file_name": f"{new_img_id}.jpg", "width": w, "height": h})
                license_annotations.extend(license_in_tw)

    # Save updated COCO annotation files
    helmet_coco_data = {
        "images": new_images_helmet,
        "annotations": helmet_annotations,
        "categories": [c for c in coco_data["categories"] if c["id"] == helmet_category_id]
    }

    license_coco_data = {
        "images": new_images_license,
        "annotations": license_annotations,
        "categories": [c for c in coco_data["categories"] if c["id"] == license_plate_category_id]
    }

    with open(os.path.join(output_folder_helmet, "_annotations.json"), "w") as f:
        json.dump(helmet_coco_data, f, indent=4)

    with open(os.path.join(output_folder_license, "_annotations.json"), "w") as f:
        json.dump(license_coco_data, f, indent=4)

    print(f"Processing complete! Helmet data in '{output_folder_helmet}', License Plate data in '{output_folder_license}'.")


In [12]:
extract_twowheeler_data(
    original_annotation_file="Data/train/_annotations.coco.json",
    image_folder="Data/train",
    output_folder_helmet="Data_helmet/train",
    output_folder_license="Data_license/train"
)

Processing complete! Helmet data in 'Data_helmet/train', License Plate data in 'Data_license/train'.


In [13]:
extract_twowheeler_data(
    original_annotation_file="Data/test/_annotations.coco.json",
    image_folder="Data/test",
    output_folder_helmet="Data_helmet/test",
    output_folder_license="Data_license/test"
)

Processing complete! Helmet data in 'Data_helmet/test', License Plate data in 'Data_license/test'.


In [14]:
extract_twowheeler_data(
    original_annotation_file="Data/valid/_annotations.coco.json",
    image_folder="Data/valid",
    output_folder_helmet="Data_helmet/valid",
    output_folder_license="Data_license/valid"
)

Processing complete! Helmet data in 'Data_helmet/valid', License Plate data in 'Data_license/valid'.
